In [74]:
import polars as pl
import os
import json
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath('')).resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]  # Fast_Data_Discovery

exp_path = NOTEBOOK_DIR / 'logs5'
base_scores = pl.read_csv(PROJECT_ROOT / 'experiments' / 'base_tables' / 'base_simple_scores.csv').rename({'': 'table'})
aug_scores = pl.read_csv(exp_path / 'simple_results.csv').rename({'_duplicated_0': 'experiment'})
aug_scores = aug_scores.with_columns(pl.col('experiment').fill_null(pl.col('')), pl.col(['roc_auc', 'f1']).cast(pl.Float64))

exp_dir = os.listdir(exp_path)
runtime_res = {}
for exp in exp_dir:
    if not os.path.isdir(exp_path / exp):
        continue
    comb_log = exp_path / exp / f'{exp}.log'
    with open(comb_log) as f:
        lines = f.readlines()
    for i in range(len(lines)):
        lines[i] = json.loads(lines[i])
    has_error = any(line.get('level') == 'ERROR' for line in lines)
    df = pl.from_records(lines, orient='col')
    if has_error:
        runtime_res[exp] = None
    else:
        runtime = df.select(pl.col('runtime').sum()).to_series()[0]
        runtime_res[exp] = runtime
runtime_df = pl.from_dicts([{'experiment': k, 'runtime': v} for k, v in runtime_res.items()])
aug_scores = aug_scores.join(runtime_df, left_on='experiment', right_on='experiment', how='left')
aug_scores = aug_scores.with_columns(
    pl.col('experiment').str.split('_').list.to_struct(upper_bound=3, fields=['lake', 'table', 'algorithm']).struct.unnest()
)
aug_scores = aug_scores.join(base_scores, on='table', how='left', suffix='_base')

aug_scores = aug_scores.with_columns((pl.col('rmse_base').fill_null(0) + pl.col('f1_weighted_base').fill_null(0)).round(3).alias('score_base'))
aug_scores = aug_scores.with_columns((pl.col('rmse').fill_null(0) + pl.col('f1_weighted').fill_null(0)).round(3).alias('score'))
aug_scores = aug_scores.with_columns(
    pl.when(pl.col('rmse').is_not_null())
    .then(pl.lit('rmse'))
    .otherwise(pl.lit('f1'))
    .alias('metric')
)
# Filter out housing dataset
aug_scores = aug_scores.filter(pl.col('table').is_in(['housing', 'inspections']).not_())
aug_scores = aug_scores.with_columns(
    pl.when(
        (pl.col('algorithm') == 'qcr') &
        (pl.col('table').is_in(['arrest', 'food', 'hospital', 'trees']))
    )
    .then(None)
    .otherwise(pl.col('score'))
    .alias('score')
)

# Create base rows for each unique (lake, table) combination
base_rows = (
    aug_scores
    .unique(subset=['lake', 'table'])
    .with_columns([
        pl.lit('base').alias('algorithm'),
        pl.col('score_base').alias('score'),
        pl.col('rmse_base').alias('rmse'),
        pl.col('f1_weighted_base').alias('f1'),
        pl.lit(0.0).alias('runtime'),
        (pl.col('lake') + '_' + pl.col('table') + '_base').alias('experiment')
    ])
    .select(aug_scores.columns)
)
aug_scores = pl.concat([aug_scores, base_rows])
# aug_scores = aug_scores.with_columns(
#     pl.when(pl.col('table') == 'pageviews')
#     .then((pl.col('score')/1e6).round(3))
#     .otherwise(pl.col('score'))
#     .alias('score')
# )
aug_scores = aug_scores.unique('experiment').filter(pl.col('algorithm') != 'backward').filter(pl.col('table') != 'elections')

In [75]:
algorithm_order = ['base', 'arda', 'autofeat', 'kitana', 'qcr',  'forward', 'backward']

# Get base displayed score per table (already scaled for pageviews), deduplicated
base_disp = (
    aug_scores.filter(pl.col('algorithm') == 'base')
    .select('table', pl.col('score').alias('_base_disp'))
    .unique(subset=['table'])
)

# Relative improvement per (algorithm, table)
rel = (
    aug_scores.join(base_disp, on='table')
    .with_columns(
        pl.when(pl.col('metric') == 'rmse')
        .then((pl.col('_base_disp') - pl.col('score')) / pl.col('_base_disp'))
        .otherwise((pl.col('score') - pl.col('_base_disp')) / pl.col('_base_disp'))
        .alias('rel_impr')
    )
)
avg_rel = rel.group_by('algorithm').agg(
    (pl.col('rel_impr').filter(pl.col('rel_impr').is_finite()).mean() * 100).round(1).alias('Avg Δ (%)')
)

pivoted = (
    aug_scores.with_columns(pl.col('table')+' '+pl.col('metric'))
    .pivot(index='algorithm', columns='table', values='score', aggregate_function='first')
    .join(avg_rel, on='algorithm')
    .with_columns(pl.col('algorithm').cast(pl.Enum(algorithm_order)))
    .sort('algorithm')
)
pivoted

/tmp/ipykernel_3902821/486835253.py:26: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(index='algorithm', columns='table', values='score', aggregate_function='first')


algorithm,hospital f1,fire rmse,arrest f1,trees f1,imdb rmse,food f1,realestate rmse,vgsales rmse,pageviews rmse,jobs rmse,energy rmse,Avg Δ (%)
enum,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""base""",0.14,61687.319,0.498,0.654,0.559,0.525,3.1704e7,1.415,7.1426e6,2875.24,9.814,0.0
"""arda""",0.14,61687.319,0.498,0.654,0.559,0.525,3.1704e7,1.415,7.1426e6,2875.24,3.818,5.6
"""autofeat""",0.411,61687.319,0.498,0.784,0.559,0.525,3.1704e7,1.415,7.1426e6,2875.24,9.814,19.4
"""kitana""",0.14,61722.464,0.498,0.654,0.559,0.525,3.1704e7,1.415,7.1426e6,2875.24,9.797,0.0
"""qcr""",null,61806.626,null,null,0.224,null,3.2016e7,1.444,4.4711e6,3251.335,9.434,12.1
"""forward""",0.139,61765.108,0.986,0.674,0.492,0.931,3.1475e7,1.41,2.8024e6,3322.73,8.528,22.6


In [76]:
algorithm_order = ['base', 'arda', 'autofeat', 'kitana', 'qcr',  'forward', 'backward']
pivoted = (
    aug_scores.with_columns(pl.col('table'))
    .pivot(index='algorithm', columns='table', values='runtime', aggregate_function='first')
    .with_columns(pl.col('algorithm').cast(pl.Enum(algorithm_order)))
    .with_columns(pl.exclude('algorithm').round(3))
    .sort('algorithm')
    .filter(pl.col('algorithm') != 'base')
)
pivoted

/tmp/ipykernel_3902821/4064558309.py:4: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(index='algorithm', columns='table', values='runtime', aggregate_function='first')


algorithm,hospital,fire,arrest,trees,imdb,food,realestate,vgsales,pageviews,jobs,energy
enum,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""arda""",92.653,740.252,44.756,55.497,null,null,null,null,null,100.544,63.222
"""autofeat""",223.588,null,5.856,260.399,null,null,null,null,null,null,1.607
"""kitana""",32.58,795.182,4.358,18.913,100.203,151.056,0.711,126.996,123.129,9.554,0.949
"""qcr""",8.593,1149.643,8.365,1.985,3.847,1.359,325.423,15.223,1.33,5.834,229.228
"""forward""",10.921,9.476,9.853,46.114,14.768,13.26,117.396,11.726,86.124,5.764,16.43


In [77]:
import math

TABLES_DIR = PROJECT_ROOT.parent / 'Matryoshka' / 'tables'

# Dataset order matching datasets.tex; split into regression and classification
regression_datasets = ['realestate', 'energy', 'fire', 'imdb', 'jobs', 'pageviews', 'vgsales']
classification_datasets = ['arrest', 'food', 'hospital', 'trees']
all_datasets = regression_datasets + classification_datasets

# All dataset columns get a grey background
SHADE_CMD = '\\cellcolor{gray!15}'

# Display names for columns
dataset_display = {
    'realestate': 'Real Estate', 'energy': 'Energy',
    'fire': 'Fire Incidents', 'imdb': 'IMDB', 'jobs': 'Jobs',
    'pageviews': 'Page Views', 'vgsales': 'VG Sales',
    'arrest': 'Arrest', 'food': 'Food', 'hospital': 'Hospital', 'trees': 'Trees',
}

# Algorithm display names
algo_display = {
    'base': 'Base', 'arda': 'ARDA', 'autofeat': 'AutoFeat',
    'kitana': 'Kitana', 'qcr': 'QCR',
    'forward': '\\system', 'backward': 'Bwd (\\system)',
}

def fmt_number(val, is_large=False):
    """Format a number: add thousand separators for large integers, otherwise 3 decimals."""
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return '---'
    if is_large:
        return f'{int(round(val)):,}'.replace(',', '\\,')
    return f'{val:.3f}'

def highlight(val_str, rank):
    """Apply bold (best) or underline (second-best) formatting."""
    if rank == 1:
        return f'\\textbf{{{val_str}}}'
    elif rank == 2:
        return f'\\underline{{{val_str}}}'
    return val_str

def compute_ranks(series, lower_is_better):
    """Return rank 1 (best) and 2 (second-best) for each value in the series."""
    vals = series.to_list()
    valid = [(v, i) for i, v in enumerate(vals) if v is not None and not (isinstance(v, float) and math.isnan(v))]
    if not valid:
        return [0] * len(vals)
    sorted_vals = sorted(set(v for v, _ in valid), reverse=not lower_is_better)
    best_val = sorted_vals[0]
    second_val = sorted_vals[1] if len(sorted_vals) > 1 else None
    ranks = []
    for v in vals:
        if v is None or (isinstance(v, float) and math.isnan(v)):
            ranks.append(0)
        elif v == best_val:
            ranks.append(1)
        elif second_val is not None and v == second_val:
            ranks.append(2)
        else:
            ranks.append(0)
    return ranks

def is_large_value(dataset, table_type):
    """Determine if a dataset column needs thousand separators."""
    if table_type == 'runtime':
        return False
    return dataset in ('realestate', 'fire', 'jobs')

def compute_avg_rel_improvement(pivoted_df, algo_order):
    """Compute average relative improvement (%) vs base for each algorithm."""
    base_disp = (
        aug_scores.filter(pl.col('algorithm') == 'base')
        .select('table', pl.col('score').alias('_base_disp'))
        .unique(subset=['table'])
    )
    rel = (
        aug_scores.join(base_disp, on='table')
        .with_columns(
            pl.when(pl.col('metric') == 'rmse')
            .then((pl.col('_base_disp') - pl.col('score')) / pl.col('_base_disp'))
            .otherwise((pl.col('score') - pl.col('_base_disp')) / pl.col('_base_disp'))
            .alias('rel_impr')
        )
    )
    # Exclude QCR classification results from its average
    rel = rel.filter(
        ~((pl.col('algorithm') == 'qcr') & pl.col('table').is_in(classification_datasets))
    )
    return rel.group_by('algorithm').agg(
        (pl.col('rel_impr').filter(pl.col('rel_impr').is_finite()).mean() * 100).round(1).alias('avg_rel_delta')
    )

def compute_geo_mean_runtime(aug_scores, classification_datasets):
    """Compute geometric mean of runtimes for each algorithm (excluding base)."""
    runtimes = aug_scores.filter(pl.col('algorithm') != 'base')
    # Exclude QCR classification datasets (no results there)
    runtimes = runtimes.filter(
        ~((pl.col('algorithm') == 'qcr') & pl.col('table').is_in(classification_datasets))
    )
    return runtimes.group_by('algorithm').agg(
        pl.col('runtime').filter(
            pl.col('runtime').is_not_null() & pl.col('runtime').is_finite() & (pl.col('runtime') > 0)
        ).log().mean().exp().round(1).alias('avg_rel_delta')
    )

def generate_latex_table(pivoted_df, table_type, algo_order, avg_rel_df=None, avg_lower_is_better=False, error_mask=None):
    """Generate a LaTeX table string from a pivoted Polars DataFrame.
    
    table_type: 'score' or 'runtime'
    avg_rel_df: optional DataFrame with 'algorithm' and 'avg_rel_delta' columns
    avg_lower_is_better: if True, lower avg values are ranked better (e.g. for runtime)
    error_mask: set of (algorithm, dataset) tuples to display as '---'
    """
    if error_mask is None:
        error_mask = set()
    n_reg = len(regression_datasets)
    n_cls = len(classification_datasets)
    n_total = n_reg + n_cls
    has_avg = avg_rel_df is not None
    
    # For scores, column names include metric suffix; for runtime, just dataset name
    if table_type == 'score':
        metric_map = dict(aug_scores.select('table', 'metric').unique().iter_rows())
        col_map = {}
        for ds in all_datasets:
            metric = metric_map.get(ds, '')
            col_name = f'{ds} {metric}'
            col_map[ds] = col_name
        lower_is_better = {ds: (metric_map.get(ds) == 'rmse') for ds in all_datasets}
    else:
        col_map = {ds: ds for ds in all_datasets}
        lower_is_better = {ds: True for ds in all_datasets}
    
    df = pivoted_df
    if has_avg:
        df = df.join(avg_rel_df, on='algorithm', how='left')
    
    df = df.with_columns(
        pl.col('algorithm').cast(pl.Enum(algo_order))
    ).sort('algorithm')
    
    if has_avg:
        avg_ranks = compute_ranks(df['avg_rel_delta'], lower_is_better=avg_lower_is_better)
    
    # Compute ranks per column (exclude QCR for classification datasets)
    algos_list = [str(a) for a in df['algorithm'].to_list()]
    ranks = {}
    for ds in all_datasets:
        col = col_map.get(ds)
        if col and col in df.columns:
            if ds in classification_datasets:
                # Mask QCR values so they don't participate in ranking
                masked = [None if algos_list[i] == 'qcr' else df[col][i] for i in range(len(df))]
                ranks[ds] = compute_ranks(pl.Series(masked), lower_is_better[ds])
            else:
                ranks[ds] = compute_ranks(df[col], lower_is_better[ds])
        else:
            ranks[ds] = [0] * len(df)
    
    # Build header
    lines = []
    lines.append('\\begin{table*}[t]')
    lines.append('    \\small')
    lines.append('    \\centering')
    lines.append('    \\setlength\\tabcolsep{3pt}')
    
    # Avg Δ% column goes right after Method (col 2), then regression, then classification
    if table_type == 'score':
        lines.append('    \\caption{Downstream prediction quality. Regression tasks are evaluated with RMSE ($\\downarrow$), classification tasks with weighted F1 ($\\uparrow$). Best results are in \\textbf{bold}, second-best are \\underline{underlined}.}')
    else:
        lines.append('    \\caption{End-to-end runtime in seconds. Best results are in \\textbf{bold}, second-best are \\underline{underlined}. Dashes (---) indicate that the method did not run end-to-end due to convergence error (all methods except QCR). QCR was not applicable for classification tasks.}')
    
    lines.append('    \\vspace{-0.3cm}')
    lines.append('')
    
    avg_col_spec = 'r|' if has_avg else ''
    lines.append(f'    \\begin{{tabular}}{{l|{avg_col_spec}{"r" * n_reg}|{"r" * n_cls}}}')
    lines.append('        \\toprule')
    
    # Multicolumn spans shift by 1 when avg column is present
    if has_avg:
        reg_start = 3
    else:
        reg_start = 2
    reg_end = reg_start + n_reg - 1
    cls_start = reg_end + 1
    cls_end = cls_start + n_cls - 1
    
    if table_type == 'score':
        lines.append(f'        & & \\multicolumn{{{n_reg}}}{{c|}}{{\\textit{{Regression (RMSE $\\downarrow$)}}}} & \\multicolumn{{{n_cls}}}{{c}}{{\\textit{{Classification (F1 $\\uparrow$)}}}} \\\\' if has_avg else
                      f'        & \\multicolumn{{{n_reg}}}{{c|}}{{\\textit{{Regression (RMSE $\\downarrow$)}}}} & \\multicolumn{{{n_cls}}}{{c}}{{\\textit{{Classification (F1 $\\uparrow$)}}}} \\\\')
    else:
        lines.append(f'        & & \\multicolumn{{{n_reg}}}{{c|}}{{\\textit{{Regression}}}} & \\multicolumn{{{n_cls}}}{{c}}{{\\textit{{Classification}}}} \\\\' if has_avg else
                      f'        & \\multicolumn{{{n_reg}}}{{c|}}{{\\textit{{Regression}}}} & \\multicolumn{{{n_cls}}}{{c}}{{\\textit{{Classification}}}} \\\\')
    
    lines.append(f'        \\cmidrule(lr){{{reg_start}-{reg_end}}} \\cmidrule(l){{{cls_start}-{cls_end}}}')
    
    # Column headers
    header = '        \\textbf{Method}'
    if has_avg:
        if table_type == 'score':
            header += '\n            & \\textbf{Avg~$\\Delta$\\%}'
        else:
            header += '\n            & \\textbf{Geom.~Mean}'
    for ds in all_datasets:
        header += f'\n            & {SHADE_CMD}\\textbf{{{dataset_display[ds]}}}'
    header += ' \\\\'
    lines.append(header)
    lines.append('        \\midrule')
    
    # Data rows
    algos = df['algorithm'].to_list()
    system_algos = {'forward', 'backward'}
    
    for row_idx, algo in enumerate(algos):
        algo_str = str(algo)
        display_name = algo_display.get(algo_str, algo_str)
        
        if algo_str in system_algos and (row_idx == 0 or str(algos[row_idx - 1]) not in system_algos):
            lines.append('        \\midrule')
        
        # Build avg cell first (goes right after method)
        if has_avg:
            avg_val = df['avg_rel_delta'][row_idx]
            if avg_val is not None and not (isinstance(avg_val, float) and math.isnan(avg_val)):
                if algo_str == 'base':
                    avg_cell = '---'
                elif table_type == 'score':
                    sign = '+' if avg_val >= 0 else ''
                    avg_cell = f'{sign}{avg_val:.1f}'
                    avg_cell = highlight(avg_cell, avg_ranks[row_idx])
                else:
                    avg_cell = f'{avg_val:.1f}'
                    avg_cell = highlight(avg_cell, avg_ranks[row_idx])
            else:
                avg_cell = '---'
        
        # Build dataset cells
        cells = []
        for ds_idx, ds in enumerate(all_datasets):
            col = col_map.get(ds)
            # QCR has no classification results
            if algo_str == 'qcr' and ds in classification_datasets:
                val_str = '---'
            elif (algo_str, ds) in error_mask:
                val_str = '---'
            elif col and col in df.columns:
                val = df[col][row_idx]
                large = is_large_value(ds, table_type)
                val_str = fmt_number(val, is_large=large)
                rank = ranks[ds][row_idx]
                val_str = highlight(val_str, rank)
            else:
                val_str = '---'
            val_str = f'{SHADE_CMD}{val_str}'
            cells.append(val_str)
        
        reg_cells = ' & '.join(cells[:n_reg])
        cls_cells = ' & '.join(cells[n_reg:])
        lines.append(f'        {display_name}')
        if has_avg:
            lines.append(f'            & {avg_cell}')
        lines.append(f'            & {reg_cells}')
        lines.append(f'            & {cls_cells} \\\\')
    
    lines.append('        \\bottomrule')
    lines.append('    \\end{tabular}')
    lines.append('')
    label = 'scores' if table_type == 'score' else 'runtime'
    lines.append(f'    \\label{{table:{label}}}')
    lines.append('\\end{table*}')
    
    return '\n'.join(lines)

# --- Build error mask: (algorithm, table) pairs where runtime is None (pipeline errored) ---
error_mask = set(
    aug_scores.filter(
        pl.col('runtime').is_null() & (pl.col('algorithm') != 'base')
    ).select('algorithm', 'table').iter_rows()
)

# --- Build the two pivoted dataframes ---
score_pivoted = (
    aug_scores.with_columns(pl.col('table') + ' ' + pl.col('metric'))
    .pivot(index='algorithm', columns='table', values='score', aggregate_function='first')
)
runtime_pivoted = (
    aug_scores.with_columns(pl.col('table'))
    .pivot(index='algorithm', columns='table', values='runtime', aggregate_function='first')
    .with_columns(pl.exclude('algorithm').round(3))
    .filter(pl.col('algorithm') != 'base')
)

# --- Compute avg relative improvement ---
avg_rel_df = compute_avg_rel_improvement(score_pivoted, algorithm_order)

# --- Compute geometric mean of runtimes ---
geo_mean_df = compute_geo_mean_runtime(aug_scores, classification_datasets)

# --- Generate and save ---
score_tex = generate_latex_table(score_pivoted, 'score', algorithm_order, avg_rel_df=avg_rel_df, error_mask=error_mask)
runtime_tex = generate_latex_table(runtime_pivoted, 'runtime', [a for a in algorithm_order if a != 'base'], avg_rel_df=geo_mean_df, avg_lower_is_better=True)

(TABLES_DIR / 'performance.tex').write_text(score_tex + '\n')
(TABLES_DIR / 'efficiency.tex').write_text(runtime_tex + '\n')

/tmp/ipykernel_3902821/1717326541.py:287: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(index='algorithm', columns='table', values='score', aggregate_function='first')
/tmp/ipykernel_3902821/1717326541.py:291: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(index='algorithm', columns='table', values='runtime', aggregate_function='first')


3356